# Phase 8 — AI 계측 코드 생성 (예제)

AI 로 코드를 빠르게 만들되, **실행해서 알려진 값과 대조**하는 검증 과정을 익힙니다.

## 0. 그래프 한글 폰트 설정

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
for _n in ['Malgun Gothic','NanumGothic','AppleGothic','Noto Sans CJK KR','Noto Sans KR']:
    if _n in {f.name for f in fm.fontManager.ttflist}:
        plt.rcParams['font.family'] = _n; break
plt.rcParams['axes.unicode_minus'] = False

## 1. 검증용 예제 이미지 (알려진 정답)
AI 코드를 검증하려면 **정답을 아는 시료**가 필요합니다. 두께 35px 인 밝은 층 이미지를 만듭니다. scale=0.25 이므로 정답 두께는 35 × 0.25 = 8.75 nm 입니다.

In [ ]:
import numpy as np, cv2
np.random.seed(3)
H,W=220,360; img=np.full((H,W),90,float); img[95:130,:]=185
img=np.clip(img+np.random.normal(0,8,(H,W)),0,255).astype(np.uint8)
scale=0.25
print('참 두께: 35 px =', 35*scale, 'nm')

## 2. 좋은 프롬프트 예시
아래 프롬프트를 사내 승인 도구(ChatGPT Enterprise)에 넣으면 아래와 비슷한 코드를 받습니다.

```
TEM 흑백 이미지에서 밝은 층의 두께를 재는 파이썬 함수를 작성해줘.
- 입력: uint8 grayscale 이미지 img, scale(nm/pixel), 측정할 열 x
- 방법: img[:, x] 세로 프로파일에서 (min+max)/2 를 지나는
        첫/마지막 위치를 선형 보간으로 찾아 두 엣지의 거리(px) × scale
- 출력: 두께(nm). 엣지를 못 찾으면 None
- 라이브러리: numpy 만 사용
- 검증: 두께 35px, scale 0.25 이면 약 8.75 가 나와야 함
```

## 3. AI 가 생성한 코드 (받았다고 가정)
받은 코드를 그대로 노트북에 붙여 실행해 봅니다.

In [ ]:
# --- AI 생성 코드 (버전 1) ---
def measure_thickness_ai(img, scale, x):
    prof = img[:, x].astype(float)
    level = (prof.min() + prof.max()) / 2
    cross = []
    for i in range(len(prof)-1):
        y0, y1 = prof[i], prof[i+1]
        if (y0 - level) * (y1 - level) < 0:
            t = (level - y0) / (y1 - y0)
            cross.append(i + t)
    if len(cross) < 2:
        return None
    return (cross[-1] - cross[0]) * scale

print(measure_thickness_ai(img, scale, x=180))

## 4. 검증: 알려진 값과 대조
정답(8.75nm)과 비교합니다. 오차가 작으면 신뢰할 수 있습니다.

In [ ]:
val = measure_thickness_ai(img, scale, x=180)
print('측정값 :', round(val,3), 'nm  / 정답: 8.75 nm')
assert abs(val - 8.75) < 0.3, 'AI 코드 결과가 정답과 다릅니다!'
print('검증 통과 — 이 코드는 신뢰할 수 있습니다')

## 5. 버그가 숨은 AI 코드 (흔한 함정: scale 누락)
AI 가 아래처럼 **scale 을 곱하지 않은** 코드를 줄 때가 있습니다. 그럴듯해 보이지만 단위가 틀립니다. 실행 → 정답과 대조하면 바로 드러납니다.

In [ ]:
# --- AI 생성 코드 (버그: 단위 환산 누락) ---
def measure_buggy(img, scale, x):
    prof = img[:, x].astype(float)
    level = (prof.min() + prof.max()) / 2
    cross = []
    for i in range(len(prof)-1):
        if (prof[i]-level)*(prof[i+1]-level) < 0:
            t=(level-prof[i])/(prof[i+1]-prof[i]); cross.append(i+t)
    return cross[-1] - cross[0]        # <-- scale 을 안 곱함 (px 그대로)

bad = measure_buggy(img, scale, x=180)
print('버그 코드 결과 :', round(bad,2), '(정답 8.75 과 크게 다름 -> px 를 반환 중)')

## 6. 버그 수정
검증에서 드러난 문제(단위)를 고칩니다. 프롬프트에 "× scale 을 반드시 포함"이라고 다시 요청해도 됩니다.

In [ ]:
def measure_fixed(img, scale, x):
    px = measure_buggy(img, scale, x)      # px 두께
    return px * scale                      # nm 로 환산 (수정)

fixed = measure_fixed(img, scale, x=180)
assert abs(fixed - 8.75) < 0.3
print('수정 후 :', round(fixed,3), 'nm — 검증 통과')

## 7. 검증 체크리스트
AI 코드를 받으면 항상 확인합니다:
1. 에러 없이 실행되는가 (존재하지 않는 함수·환각 여부)
2. 결과가 상식적 범위인가
3. **알려진 시료로 정답과 대조**했는가
4. 단위 환산(× scale), 색 순서(BGR/RGB), 예외 처리가 맞는가
5. 측정 원리가 의도한 방법과 같은가

---
예제 실행을 마친 후 `08_practice.ipynb` 의 문제를 해결하십시오. 프롬프트 템플릿은 `08_prompt_templates.md` 참고.